In [29]:
import os
import mlflow
import torch
import random
import numpy as np
from tqdm import tqdm
from os.path import join as pjoin
from datetime import datetime
import sys
from torchsummary import summary
import pickle as pkl
import pylops
from src.steinbrinker.BPDN import BPDN

PROJECT_PATH = 'C:\\Users\\galig\\Desktop\\ecg-denoising\\workspace'
PROJECT_SRC_PATH = os.path.join(PROJECT_PATH, 'src\\refactored')

sys.path.append(PROJECT_SRC_PATH)

from src.trainers.FISTANetTrainer import FISTANetTrainer
from src.models.FISTANet import FISTANet
from src.trainers.CNNTrainer import CNNTrainer
from src.models.CNN import CNN
from src.utils.loader import DataSplit

RANDOM_SEED = 42

mlflow.set_tracking_uri('http://localhost:8080')

In [4]:
DATA_DIR = os.path.join(PROJECT_PATH, 'data')
DATA_FILE_GEN = 'generated/BW_master_10000_2024-04-07-12-43-32.pkl'
DATA_FILE_SIGS = 'steinbrinker/testing_data_mvg_avg.npy'
DATA_FILE_BW = 'mit-bih/bw'
DATA_FILE_GAUSS = 'generated/gaussian_noise.npy'
DATA_FILE_BPDN_MAX = 'generated/BW_alphas-BPDN_10000_2024-04-07-12-43-32.npy'
DATA_FILE_BPDN = 'generated/BW_alphas-BPDN-1iters_10000_2024-04-07-12-43-32.npy'
# DATA_FILE_BPDN = 'generated/BW_alphas-BPDN-3iters_10000_2024-04-07-12-43-32.npy'
# DATA_FILE_BPDN = 'generated/BW_alphas-BPDN-5iters_10000_2024-04-07-12-43-32.npy'
DATA_FILE_BPDN_FINAL = 'generated/BW_alphas-BPDN_10000_2024-04-07-12-43-32.npy'
DICT_FILE_BW = 'steinbrinker/dictionary_BW_real_data.npy'
NOISE_TYPE = 'bw'
if NOISE_TYPE == 'bw':
    DATA_FILE_NOISE = DATA_FILE_BW
elif NOISE_TYPE == 'gauss':
    DATA_FILE_NOISE = DATA_FILE_GAUSS
DATA_SIZE = 10000
BATCH_SIZE = 1000
TVT_SPLIT = {
    'train': 80,
    'valid': 10,
    'test': 10
}

FNET_LAYER_NO = 4
FNET_FEATURE_NO = 16
LAMBDA_PRED_SP_LOSS = 1
LAMBDA_SYM_LOSS = 1e-3
LAMBDA_SP_LOSS = 1e-2

LR_DEC_AFTER = 15000
LR_DEC_EVERY = 10
LEARNING_RATE = 1e-3

In [10]:
# Seed random generators to ensure deterministic experiments
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
generator = torch.Generator()
generator.manual_seed(RANDOM_SEED) 

# Define PyTorch device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('cpu')

# Read and log train, validation and test datasets
trn_ldr, val_ldr, tst_ldr = DataSplit(DATA_DIR, NOISE_TYPE, DATA_FILE_GEN, DATA_FILE_SIGS, DATA_FILE_NOISE,
                                      DATA_FILE_BPDN, DATA_FILE_BPDN_FINAL, TVT_SPLIT, BATCH_SIZE, generator=generator)
Psi = torch.from_numpy(np.load(pjoin(DATA_DIR, DICT_FILE_BW)))
Psi = Psi.clone().detach().to(device=device)
bpdn_est = np.load(os.path.join(DATA_DIR, DATA_FILE_BPDN_MAX))
dictionary = np.load(os.path.join(DATA_DIR, DICT_FILE_BW))

# Specify and log training parameters
dt = datetime.now().strftime('%Y-%m-%d-%H-%M-%S')
params = {
    'device': device,
    'batch_size': BATCH_SIZE,
    'lr': LEARNING_RATE,
    'lr_dec_after': LR_DEC_AFTER,
    'lr_dec_every': LR_DEC_EVERY,
    'fnet_layer_no': FNET_LAYER_NO,
    'fnet_feature_no': FNET_FEATURE_NO,
    'lambda_sp_loss': LAMBDA_SP_LOSS,   
    'lambda_pred_sp_loss': LAMBDA_PRED_SP_LOSS,
    'lambda_sym_loss': LAMBDA_SYM_LOSS,
}

In [12]:
params['load_model_run'] = 'a7f7899e0ad043c68319cea0e6a7d548'
params['load_model_epoch'] = 17000

model = FISTANet(FNET_LAYER_NO, FNET_FEATURE_NO)

# Load model
model = model.to(device)

# Print model summary
summary(model, [(100, 1), (2500, 1), (2500, 100)], verbose=1)

# Define trainer and evaluate model
trainer = FISTANetTrainer(model, Psi, bpdn_est, dictionary, params)

Layer (type:depth-idx)                   Output Shape              Param #
├─BasicBlock: 1-1                        [-1, 1, 100, 1]           --
├─ModuleList: 1                          []                        --
|    └─BasicBlock: 2-1                   [-1, 1, 100, 1]           (recursive)
├─BasicBlock: 1                          []                        --
|    └─Softplus: 2-2                     [-1]                      --
├─ModuleList: 1                          []                        --
|    └─BasicBlock: 2                     []                        --
|    |    └─Softplus: 3-1                [-1]                      --
├─BasicBlock: 1                          []                        --
|    └─Conv2d: 2-3                       [-1, 16, 100, 1]          160
├─ModuleList: 1                          []                        --
|    └─BasicBlock: 2                     []                        --
|    |    └─Conv2d: 3-2                  [-1, 16, 100, 1]          (recursi

In [14]:
x = trainer.infer(tst_ldr)
%timeit -r5 -n3 x = trainer.infer(tst_ldr)

491 ms ± 8.39 ms per loop (mean ± std. dev. of 5 runs, 3 loops each)


In [11]:
params['load_model_run'] = '9167f4ead13343e89df56bfdc20473e5'
params['load_model_epoch'] = 3000

model = CNN()

# Load model
model = model.to(device)

# Print model summary
summary(model, (1, 2500), verbose=1)

# Define trainer and evaluate model
trainer = CNNTrainer(model, Psi, bpdn_est, dictionary, params)

Layer (type:depth-idx)                   Output Shape              Param #
├─Sequential: 1-1                        [-1, 36, 39]              --
|    └─Conv1d: 2-1                       [-1, 36, 2500]            720
|    └─BatchNorm1d: 2-2                  [-1, 36, 2500]            72
|    └─ReLU: 2-3                         [-1, 36, 2500]            --
|    └─AvgPool1d: 2-4                    [-1, 36, 1250]            --
|    └─Conv1d: 2-5                       [-1, 36, 1250]            24,660
|    └─BatchNorm1d: 2-6                  [-1, 36, 1250]            72
|    └─ReLU: 2-7                         [-1, 36, 1250]            --
|    └─AvgPool1d: 2-8                    [-1, 36, 625]             --
|    └─Conv1d: 2-9                       [-1, 36, 625]             24,660
|    └─BatchNorm1d: 2-10                 [-1, 36, 625]             72
|    └─ReLU: 2-11                        [-1, 36, 625]             --
|    └─AvgPool1d: 2-12                   [-1, 36, 312]             --
|    └

In [12]:
x = trainer.infer(tst_ldr)
%timeit -r5 -n3 x = trainer.infer(tst_ldr)

21 s ± 72.1 ms per loop (mean ± std. dev. of 5 runs, 3 loops each)


In [15]:
VALID_INP_SIGS = 362
INP_SMPL_FREQ = 250

inp_sigs = np.load('data/steinbrinker/testing_data_mvg_avg.npy')[:, :VALID_INP_SIGS]
dictionary = np.load('data/steinbrinker/dictionary_BW_real_data.npy')
bw = np.load('data/generated/BW_noise.npy')

file_name = 'BW_master_10000_2024-04-07-12-43-32.pkl'
with open(f'data/generated/{file_name}', 'rb') as handle:
    gen_data = pkl.load(handle)

gen_data_size = gen_data['params']['GEN_DATA_SIZE']
inp_smpl_freq = gen_data['params']['INP_SMPL_FREQ']
gen_sig_secs = gen_data['params']['GEN_SIG_SECS']
valid_inp_sigs = gen_data['params']['VALID_INP_SIGS']
gen_sig_cuts = np.int_(gen_data['data'])
gen_sig_len = inp_smpl_freq * gen_sig_secs
lambda_param = 0.01
max_iters = 10000

start_idx = 9000
end_idx = 10000

targets = np.zeros((end_idx - start_idx, dictionary.shape[0]))
noises = np.zeros((end_idx - start_idx, dictionary.shape[0]))
for i in tqdm(range(end_idx - start_idx)):
    targets[i, :] = inp_sigs[gen_sig_cuts[1, start_idx+i]:gen_sig_cuts[1, start_idx+i]+gen_sig_len, gen_sig_cuts[0, start_idx+i]]
    noises[i, :] = bw[gen_sig_cuts[2, start_idx+i]:gen_sig_cuts[2, start_idx+i]+gen_sig_len]

100%|███████████████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 21360.60it/s]


In [51]:
%%timeit -r5 -n3 
ixs = random.sample(list(range(targets.shape[0])), targets.shape[0])[:100]
x = [pylops.optimization.sparsity.fista(pylops.MatrixMult(dictionary), targets[i] + noises[i], niter=100, eps=5e-1, tol=1e-3) for i in ixs]

4.84 s ± 17.3 ms per loop (mean ± std. dev. of 5 runs, 3 loops each)


In [30]:
%%timeit -r5 -n3 
ixs = random.sample(list(range(targets.shape[0])), targets.shape[0])[:10]
x = [pylops.optimization.sparsity.fista(pylops.MatrixMult(dictionary), targets[i] + noises[i], niter=100, eps=5e-1, tol=1e-3) for i in ixs]

7.27 s ± 37.5 ms per loop (mean ± std. dev. of 5 runs, 3 loops each)


In [52]:
%%timeit -r5 -n3
ixs = random.sample(list(range(targets.shape[0])), targets.shape[0])[:1]
x = [BPDN(targets[i, :] + noises[i, :], dictionary, lambda_param=lambda_param, max_iters=max_iters)[1] for i in ixs]

8.54 s ± 425 ms per loop (mean ± std. dev. of 5 runs, 3 loops each)


In [55]:
%%timeit -r5 -n3
ixs = random.sample(list(range(targets.shape[0])), targets.shape[0])[:3]
x = [BPDN(targets[i, :] + noises[i, :], dictionary, lambda_param=lambda_param, max_iters=max_iters)[1] for i in ixs]

23.9 s ± 539 ms per loop (mean ± std. dev. of 5 runs, 3 loops each)


In [54]:
%%timeit -r5 -n3
ixs = random.sample(list(range(targets.shape[0])), targets.shape[0])[:10]
x = [BPDN(targets[i, :] + noises[i, :], dictionary, lambda_param=lambda_param, max_iters=max_iters)[1] for i in ixs]

1min 19s ± 1.43 s per loop (mean ± std. dev. of 5 runs, 3 loops each)
